# Deep Agents 101: Journal Agent

This notebook builds one agent, one step at a time. Slides cover the concepts (harness, tools, HITL); this notebook covers the implementation.

For topics not covered today, see the self-paced LangChain Academy Deep Agents course.

## Setup: connect a model

**What you'll do:** install the SDKs and provide a model key.

In [ ]:
%pip install -q deepagents langchain-openai langgraph tavily-python dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# To use an Anthropic/OpenAI/etc. key instead, add api key to .env

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

# To use a different key, replace the block above, for example:
# from langchain.chat_models import init_chat_model
# model = init_chat_model("anthropic:claude-haiku-4-5")

## 1: The harness, what you get before writing any tool code

**What you'll learn:** what built-in read/write tools and planning capability a deep agent already has, with zero custom code.

**The payoff:** knowing the default baseline prevents rebuilding capabilities that already ship by default.

Every deep agent starts the same way: a model wrapped in a harness that already knows how to read and write files, plan, and call tools. No tools are added yet. The next cell shows what it can already do.

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(model=model)

result = agent.invoke({"messages": [{"role": "user", "content": (
    "Start a journal.md file. Log a new dated entry from these notes:\n"
    "- What I learned today: a caching bug was hiding in the retry logic\n"
    "- How I felt: relieved to ship it, a little anxious about production traffic\n"
    "- What's next: write better tests before touching anything else\n"
    "Then read the file back to me."
)}]})
for m in result['messages']:
    m.pretty_print()

## 2: Set its role with a system prompt

**What you'll learn:** how one `system_prompt` string controls the voice the agent writes in, on top of whatever facts you give it.

There is no system prompt yet. The `system_prompt` value passed to the agent is the entire prompt sent to the model.

The agent still writes the entry itself: it takes your notes as raw facts and composes them into sentences (based on the persona you give it).

In [ ]:
# Try a different persona by uncommenting one of these (or write your own):
# system_prompt = "You are a pirate. Answer only in pirate speak."
# system_prompt = "You are a toddler. Explain everything like you're five."
system_prompt = "You are a melodramatic Victorian child. Narrate everything with excessive despair and flowery, dramatic language."

agent = create_deep_agent(model=model, system_prompt=system_prompt)

result = agent.invoke({"messages": [{"role": "user", "content":
    "Log a three-sentence journal entry from these notes: what I learned today (a caching bug was "
    "hiding in the retry logic), how I felt (relieved but a little anxious), what's next (write "
    "better tests before touching anything else)."
}]})
for m in result['messages']:
    m.pretty_print()

## 3: Give it a custom tool

**What you'll learn:** how a plain Python function becomes a tool the agent can call, either picked from these below or written yourself.

**The payoff:** tools are how an agent's abilities grow past reading and writing files, this is the piece you'll customize most often.

In [ ]:
import os
import re
from collections import Counter
from langchain_core.tools import tool

@tool
def word_count(text: str) -> str:
    """Count the words in a piece of text."""
    return f"{len(text.split())} words"

@tool
def summarize_length(text: str, max_sentences: int = 2) -> str:
    """Trim a piece of text down to its first `max_sentences` sentences."""
    sentences = re.split(r"(?<=[.!?]) +", text.strip())
    return " ".join(sentences[:max_sentences])

@tool
def estimated_reading_time(text: str) -> str:
    """Estimate how long a piece of text takes to read, at 200 words per minute."""
    minutes = max(1, round(len(text.split()) / 200))
    return f"~{minutes} min read"

@tool
def keyword_extractor(text: str, top_n: int = 3) -> str:
    """Pull out the most frequent meaningful words in a piece of text."""
    stopwords = {"the", "a", "an", "and", "or", "but", "to", "of", "in", "on",
                 "for", "with", "i", "is", "it", "was", "my", "that", "this"}
    words = [w.strip(".,!?'\"").lower() for w in text.split()]
    counts = Counter(w for w in words if w not in stopwords and len(w) > 2)
    top = [word for word, _ in counts.most_common(top_n)]
    return ", ".join(top) if top else "no keywords found"

@tool
def web_search(query: str) -> str:
    """Search the web for something and return a few relevant results (title, url, short snippet)."""
    from tavily import TavilyClient
    # No signup needed for light, rate-limited use. For heavier use, get a
    # free key at tavily.com and set TAVILY_API_KEY.
    client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))
    results = client.search(query, max_results=3)["results"]
    return "\n\n".join(f"{r['title']}\n{r['url']}\n{r['content'][:200]}" for r in results)

WEATHER_CODES = {
    0: "clear sky", 1: "mainly clear", 2: "partly cloudy", 3: "overcast",
    45: "fog", 48: "depositing rime fog",
    51: "light drizzle", 53: "moderate drizzle", 55: "dense drizzle",
    61: "slight rain", 63: "moderate rain", 65: "heavy rain",
    71: "slight snow", 73: "moderate snow", 75: "heavy snow",
    80: "slight rain showers", 81: "moderate rain showers", 82: "violent rain showers",
    95: "thunderstorm", 96: "thunderstorm with slight hail", 99: "thunderstorm with heavy hail",
}

@tool
def add_weather(city: str) -> str:
    """Look up the current weather for a city, to add context to a journal entry."""
    import requests
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
    ).json()
    if not geo.get("results"):
        return f"Couldn't find a location named '{city}'."
    loc = geo["results"][0]
    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": loc["latitude"],
            "longitude": loc["longitude"],
            "current_weather": "true",
            "temperature_unit": "fahrenheit",
        },
    ).json()["current_weather"]
    condition = WEATHER_CODES.get(weather["weathercode"], "unknown conditions")
    return f"{loc['name']}: {weather['temperature']}\u00b0F, {condition}"

@tool
def on_this_day(month: int, day: int) -> str:
    """Look up a historical event that happened on a given month/day, as a journal-entry icebreaker."""
    import requests
    url = f"https://en.wikipedia.org/api/rest_v1/feed/onthisday/events/{month:02d}/{day:02d}"
    events = requests.get(url).json()["events"]
    event = events[len(events) // 2]
    return f"On this day in {event['year']}: {event['text']}"

@tool
def mood_tag(text: str) -> str:
    """Tag a journal entry with a one-word mood."""
    # Placeholder: this always returns the same canned string, it does not
    # actually read the text. A real version needs an actual sentiment
    # analysis tool, for example:
    #   - nltk's VADER: SentimentIntensityAnalyzer().polarity_scores(text)
    #   - textblob: TextBlob(text).sentiment.polarity
    # both require downloading a lexicon/corpus at runtime, which is why
    # this demo keeps it as a stub instead.
    return f"(placeholder mood tag for '{text[:30]}...', replace with a real sentiment lookup)"

TOOL_MENU = {
    "word_count": word_count,
    "summarize_length": summarize_length,
    "estimated_reading_time": estimated_reading_time,
    "keyword_extractor": keyword_extractor,
    "mood_tag": mood_tag,
    "web_search": web_search,
    "add_weather": add_weather,
    "on_this_day": on_this_day,
}

**Anatomy of a tool**: `@tool` turns a plain function into something the model can call:
- the **docstring** becomes the tool's description (how the model decides when to use it)
- the **type hints** become its input schema (what arguments it expects)
- the **return value** becomes what the model sees back

In [ ]:
chosen_tool = TOOL_MENU["word_count"]

agent = create_deep_agent(model=model, system_prompt=system_prompt, tools=[chosen_tool])

result = agent.invoke({"messages": [{"role": "user", "content":
    "Here is a journal entry: 'Today I finally shipped the feature I've been stuck on for a "
    "week. The bug turned out to be a caching issue that took forever to track down, and I "
    "ended up rewriting most of the retry logic to fix it. It feels good to have it done, "
    "though I'm a little worried about whether the fix will hold up under real traffic. "
    "Tomorrow I want to write better tests before touching anything else.' "
    "Count the words in it using your tool, then tell me what you found."
}]})
for m in result['messages']:
    m.pretty_print()

## 4: Human-in-the-loop, approve a risky action before it happens

**What this covers:** how `interrupt_on` pauses an agent mid-run so a human can approve, edit, or reject a specific tool call before it executes.

**The payoff:** any agent with access to money, message sends, or irreversible actions needs this kind of control before it is used in production.

Every tool call passes through an `Interrupt?` check. If a call matches a rule configured in `interrupt_on`, the agent pauses and hands control to a human instead of executing it directly. The human can approve the call as written, edit its arguments before it runs, or reject it outright, and the agent resumes from exactly where it paused.

This pause only works because a `checkpointer` is attached to the agent: it saves the agent's state (its messages, files, and progress so far) at the interrupt point so the run can be resumed later, potentially after the human has stepped away and come back. Resuming looks like calling `agent.invoke` again with `Command(resume={"decisions": [{"type": "approve"}]})` (or `"edit"` / `"reject"`), rather than starting a new conversation from scratch.

### Bonus: a working example (time permitting)

The cells below build a real version of the pause described above: a `share_journal_entry` tool that requires approval before it runs, using the persona and tool you already picked earlier in this notebook.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

@tool
def share_journal_entry(entry: str, platform: str) -> str:
    """Share a journal entry to an external platform. This just simulates a send, no network call is actually made."""
    return f"Shared to {platform}: {entry[:60]}..."

checkpointer = InMemorySaver()

agent = create_deep_agent(
    model=model,
    system_prompt=system_prompt,
    tools=[chosen_tool, share_journal_entry],
    interrupt_on={"share_journal_entry": True},
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "journal-hitl-demo"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": (
        "Start a journal.md file. Log a new dated entry: 'Set up human-in-the-loop "
        "approval today, it feels reassuring to have a real gate before anything gets "
        "shared externally.' Then read the file back, and share the most recent entry "
        "to the 'team-standup' platform."
    )}]},
    config=config,
)

if "__interrupt__" in result:
    request = result["__interrupt__"][0].value
    print("Paused for approval:")
    for action in request["action_requests"]:
        print(f"  {action['name']}({action['args']})")
else:
    print(result["messages"][-1].content)

The cell above paused instead of finishing, because `share_journal_entry` matched `interrupt_on`. The cell below resumes it with an approval decision.

In [ ]:
result = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)
print(result["messages"][-1].content)

## Wrap-up

You built: a filesystem-backed agent, a way to swap personas, and a custom tool.

Also covered: human-in-the-loop gating, added with a single argument.

Not covered today, but in the full LangChain Academy Deep Agents course: subagent delegation, backends (filesystem/store/composite), skills, memory across sessions, sandboxes, and deployment.